In [2]:
import pandas as pd

df = pd.read_csv("data/sales.csv")
df.head()

,sale_id,branch,city,customer_type,gender,product_name,product_category,unit_price,quantity,tax,total_price,reward_points
0,1,A,New York,Member,Male,Shampoo,Personal Care,5.50,3,1.16,17.66,1
1,2,B,Los Angeles,Normal,Female,Notebook,Stationery,2.75,10,1.93,29.43,0
2,3,A,New York,Member,Female,Apple,Fruits,1.20,15,1.26,19.26,1
3,4,A,Chicago,Normal,Male,Detergent,Household,7.80,5,2.73,41.73,0
4,5,B,Los Angeles,Member,Female,Orange Juice,Beverages,3.50,7,1.72,26.22,2


# Data Exploration (EDA)

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sale_id           1000 non-null   int64  
 1   branch            1000 non-null   str    
 2   city              1000 non-null   str    
 3   customer_type     1000 non-null   str    
 4   gender            1000 non-null   str    
 5   product_name      1000 non-null   str    
 6   product_category  1000 non-null   str    
 7   unit_price        1000 non-null   float64
 8   quantity          1000 non-null   int64  
 9   tax               1000 non-null   float64
 10  total_price       1000 non-null   float64
 11  reward_points     1000 non-null   int64  
dtypes: float64(3), int64(3), str(6)
memory usage: 93.9 KB


In [4]:
df.describe()


,sale_id,unit_price,quantity,tax,total_price,reward_points
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,10.836110,10.337000,7.758010,118.583900,6.057000
std,288.819436,5.775924,6.029908,6.538066,99.936441,9.350464
min,1.000000,1.020000,1.000000,0.080000,1.210000,0.000000
25%,250.750000,5.867500,5.000000,2.510000,38.380000,0.000000
50%,500.500000,10.615000,10.000000,5.870000,89.705000,0.000000
75%,750.250000,15.882500,16.000000,11.522500,176.072500,10.000000
max,1000.000000,20.980000,20.000000,28.390000,433.990000,43.000000


In [5]:
df.isnull().sum()


sale_id             0
branch              0
city                0
customer_type       0
gender              0
product_name        0
product_category    0
unit_price          0
quantity            0
tax                 0
total_price         0
reward_points       0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

## 1. проверим логику данных

Перед анализом всегда проверяем: правильно ли считается total_price

In [7]:
(df['unit_price'] * df['quantity'] + df['tax'] - df['total_price']).abs().max()

np.float64(5.684341886080802e-14)

total=unit_price∗quantity+tax

≈ 0 → всё корректно.

## 2 — Feature Engineering (самое важное)

создаём аналитические признаки

In [8]:
df['revenue'] = df['total_price']
df['cost'] = df['unit_price'] * df['quantity']
df['profit'] = df['revenue'] - df['cost']

df['is_member'] = (df['customer_type'] == 'Member').astype(int)
df['avg_item_price'] = df['revenue'] / df['quantity']
df['tax_percent'] = df['tax'] / df['cost']

# категории покупок
df['basket_size'] = pd.cut(
    df['quantity'],
    bins=[0,3,7,12,20],
    labels=['small','medium','large','bulk']
)

## 3 — первый настоящий бизнес-вопрос

### Кто приносит больше денег — члены программы или обычные?

In [9]:
df.groupby('customer_type')['revenue'].agg(['count','mean','sum'])

,count,mean,sum
customer_type,,,
Member,516,122.507035,63213.63
Normal,484,114.401384,55370.27


### Какие города прибыльные

In [10]:
df.groupby('city')['profit'].sum().sort_values(ascending=False)

city
Chicago        2786.02
New York       2631.75
Los Angeles    2340.24
Name: profit, dtype: float64

### Какие категории — драйвер бизнеса

In [11]:
df.groupby('product_category')['revenue'].sum().sort_values(ascending=False)

product_category
Personal Care    27050.18
Fruits           26197.45
Beverages        22983.32
Household        21615.84
Stationery       20737.11
Name: revenue, dtype: float64

## 4 — Поведенческий анализ

### Размер корзины у Member vs Normal

In [12]:
df.groupby('customer_type')['quantity'].mean()

customer_type
Member    10.459302
Normal    10.206612
Name: quantity, dtype: float64

### Дороже ли покупают Members?

In [13]:
df.groupby('customer_type')['avg_item_price'].mean()

customer_type
Member    11.719537
Normal    11.461484
Name: avg_item_price, dtype: float64

**можем ли мы по покупке понять — станет ли человек Member?**

In [14]:
df.groupby('customer_type')['revenue'].agg(['count','mean','sum'])

,count,mean,sum
customer_type,,,
Member,516,122.507035,63213.63
Normal,484,114.401384,55370.27


In [15]:
df.groupby('city')['profit'].sum()

city
Chicago        2786.02
Los Angeles    2340.24
New York       2631.75
Name: profit, dtype: float64

In [16]:
df.groupby('product_category')['revenue'].sum().sort_values(ascending=False).head(10)

product_category
Personal Care    27050.18
Fruits           26197.45
Beverages        22983.32
Household        21615.84
Stationery       20737.11
Name: revenue, dtype: float64

### 📊 Customer Analysis
#### Members vs Normal customers

| Тип клиента | Кол-во покупок | Средний чек | Общая выручка |
| ----------- | -------------- | ----------- | ------------- |
| Member      | 516            | 122.5       | 63,213        |
| Normal      | 484            | 114.4       | 55,370        |

### Вывод

#### Участники программы лояльности значительно ценнее:

- средний чек выше на ~7%

- суммарная выручка выше на ~14%

- покупают чаще

👉 Бизнес-инсайт:
**программа лояльности увеличивает доход → её нужно активно продвигать**

## 🏙️ Store Performance

| Город       | Прибыль |
| ----------- | ------- |
| Chicago     | 2786    |
| New York    | 2632    |
| Los Angeles | 2340    |

### Вывод

- Chicago — самый прибыльный филиал

- Los Angeles — отстаёт

👉 Возможные причины:

- ассортимент

- цены

- аудитория

👉 Бизнес-решение:
проверить ассортимент LA (скорее всего плохой product mix)

## 🛒 Product Performance

| Категория     | Выручка |
| ------------- | ------- |
| Personal Care | 🥇      |
| Fruits        | 🥈      |
| Beverages     | 🥉      |
| Household     |         |
| Stationery    |         |

### Вывод

- Основной драйвер — ежедневные товары

- Это НЕ супермаркет электроники → это convenience store

👉 Бизнес-решение:

- увеличить shelf space Personal Care

- делать акции на Fruits → трафик

## 🧠Поведенческий анализ (что отличает Members)

In [17]:
df.groupby('customer_type')[['quantity','avg_item_price','tax_percent']].mean()

,quantity,avg_item_price,tax_percent
customer_type,,,
Member,10.459302,11.719537,0.070013
Normal,10.206612,11.461484,0.069994


In [18]:
df.groupby(['customer_type','product_category'])['revenue'].mean().unstack()

product_category,Beverages,Fruits,Household,Personal Care,Stationery
customer_type,,,,,
Member,122.989583,130.052963,108.540748,141.014679,108.088021
Normal,122.816703,120.314158,109.911868,117.975556,101.575098


### 1) Размер корзины и цена

| Тип    | Кол-во товаров | Цена за товар |
| ------ | -------------- | ------------- |
| Member | 10.46          | 11.72         |
| Normal | 10.21          | 11.46         |

### Вывод

- Members НЕ покупают сильно больше товаров

- Они покупают **дороже товары**

👉 Значит программа лояльности не увеличивает количество покупок

👉 Она увеличивает **качество корзины (premium choices)**

Это очень типичный retail-эффект.

### 2) Где именно они тратят больше

Средний чек по категориям:

| Категория     | Member       | Normal |
| ------------- | ------------ | ------ |
| Personal Care | **141**      | 118    |
| Fruits        | **130**      | 120    |
| Beverages     | ≈ одинаково  |        |
| Household     | ≈ одинаково  |        |
| Stationery    | немного выше |        |



### Главный бизнес-инсайт 💡

**Members — это не частые покупатели.**
**Это покупатели более дорогих товаров.**

Особенно:

- Personal Care (самая большая разница)

- Fruits (качество еды)

👉 Это поведение “осознанного покупателя”, не “экономящего”.

### Что это значит для бизнеса

Реальное решение:

**Кому предлагать membership:**

- покупает Personal Care

- покупает дорогие продукты

- средний чек чуть выше среднего

👉 НЕ всем подряд
👉 только premium-поведению